# TipTop - MAVIS 2026 Hands-On - Tutorial B

In this notebook, we use TipTop to evaluate and compare possible NGS
asterisms for the MAVIS MCAO configuration, using its
[Asterism Selection features](https://astro-tiptop-services.github.io/astro-tiptop-services/docs/aquila/overview).
.

The provided `mavis_asterism.ini` file, which you can download
[here](https://astro-tiptop-services.github.io/astro-tiptop-services/resources/mavis_2026),
already contains an `[ASTERISM_SELECTION]` section defining a set of
candidate NGSs.

For this example, we use the `Singles3` mode: TipTop generates and evaluates
all possible three-star asterisms from the provided list of candidate NGSs.

All Asterism Selection modes are documented
[here](https://astro-tiptop-services.github.io/astro-tiptop-services/docs/aquila/parameterfiles#supportedmodes).


---

### A) Setup

We first import the packages needed for the asterism-selection workflow
and define the paths to the MAVIS configuration.

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tiptop.tiptop import asterismSelection

path_in = "./"
path_out = "./"

file_in = "mavis_asterism"
file_out = "mavis_asterism"

### Candidate NGSs

The candidate NGSs are defined directly in the `[ASTERISM_SELECTION]`
section of the `.ini` file.

Our MAVIS configuration provides seven candidate NGSs, shown here
together with the science-field positions.

### B) Evaluation

Seven candidate NGSs are provided here.

With `Singles3`, TipTop considers every combination of three stars,
corresponding to **35 possible asterisms**.

Let us now evaluate them with `asterismSelection()`.

In [ ]:
sr, fwhm,ee, covs, simul_ast = asterismSelection(
    simulName="MAVIS_ast",
    path2param=path_in,
    parametersFile=file_in,
    outputDir=path_out,
    outputFile=file_out,
    returnMetrics=True,
    verbose=False,
)

simul_ast.plotField(0)

### C) Asterism-selection results

TipTop has now evaluated the 35 possible NGS asterisms.

For each asterism, we can access the estimated AO performance and the
low-order penalty used to rank the different configurations.

The asterisms are ranked according to the low-order penalty:
a lower value corresponds to better predicted low-order performance.

Let us compare the 35 configurations and identify the best one.

In [ ]:
penalty = np.asarray(simul_ast.penalty_Asterism, dtype=float).squeeze()
sr_all = np.asarray(simul_ast.strehl_Asterism, dtype=float).squeeze()
fwhm_all = np.asarray(simul_ast.fwhm_Asterism, dtype=float).squeeze()

order = np.argsort(penalty)

print(
    f"{'Rank':>4}  "
    f"{'Asterism':>9}  "
    f"{'NGSs':>10}  "
    f"{'Penalty':>10}  "
    f"{'SR [%]':>8}  "
    f"{'FWHM [mas]':>11}"
)

print("-" * 65)

for rank, idx in enumerate(order, start=1):
    ngs = np.asarray(simul_ast.allAsterismsIndices[idx]) + 1
    ngs_str = "-".join(map(str, ngs))

    print(
        f"{rank:4d}  "
        f"{idx + 1:9d}  "
        f"{ngs_str:>10}  "
        f"{penalty[idx]:10.2f}  "
        f"{100 * sr_all[idx]:8.2f}  "
        f"{fwhm_all[idx]:11.2f}"
    )

### D) Reloading a previous asterism-selection run

The results of an asterism-selection run are saved to disk and can be
reloaded later without recomputing all candidate asterisms.

This is particularly useful when a large number of NGS configurations
has been evaluated.

TipTop provides `reloadAsterismSelection()` to reconstruct the simulation
object and reload the saved performance metrics.

In [ ]:
from pathlib import Path

result_files = sorted(Path(path_out).glob("MAVIS_ast*.npy"))

for file in result_files:
    print(file.name)

penalty_saved = np.load(Path(path_out) / "MAVIS_astpenalty.npy")

In [ ]:
from tiptop.tiptop import reloadAsterismSelection

sr_reload, fwhm_reload, ee_reload, covs_reload, simul_reload = (
    reloadAsterismSelection(
        simulName="MAVIS_ast",
        path2param=path_in,
        parametersFile=file_in,
        outputDir=path_out,
        outputFile=file_out,
    )
)

print('Reloaded arrays:', 
      'SR' if sr_reload is not None else '-', 
      'FW' if fwhm_reload is not None else '-', 
      'EE' if ee_reload is not None else '-')

## E) Going further — heuristic selection

In our example, seven candidate NGSs produce only 35 possible
three-star asterisms. We can therefore evaluate all of them directly
with the full TipTop model.

For larger guide-star fields, the number of possible combinations can
become much larger. TipTop also provides a heuristic approach to
accelerate this selection.

The heuristic model is first **trained on a set of fully simulated
asterisms**. Once trained, it can rapidly estimate and rank new
candidate asterisms without running the full AO simulation for every
configuration.

For multi-NGS systems such as MAVIS, the heuristic model is based on
a small neural network.

See documentation [here](https://astro-tiptop-services.github.io/astro-tiptop-services/docs/aquila/heuristic_models)

In [ ]:
import tiptop 
tiptop_path = os.path.dirname(tiptop.__file__)
print(f"Directory of 'tiptop' is: {tiptop_path}")
os.chdir(tiptop_path)

In [ ]:
# Example training workflow for a sufficiently large training dataset

from tiptop.tiptop import generateHeuristicModel
from pathlib import Path

TRAINING_INI = "ERISastRandom"
SIMUL_NAME   = "ERIStest"            # short name used as prefix for outputs
PARAMS_DIR   = tiptop_path + "/astTest"
OUTPUT_DIR   = "outputs"   
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True) 

simul_h = generateHeuristicModel(
    simulName=SIMUL_NAME,
    path2param=PARAMS_DIR,
    parametersFile=TRAINING_INI,
    outputDir=OUTPUT_DIR,
    outputFile="psf",
)

### How to reuse the model
Once trained, you can call the model to rank new asterisms instantly, without full AO simulations.

[ASTERISM_SELECTION]

heuristicModel = outputs/ERISast_hmodel.pth  ; or MAVIS..._hmodel.npy